# STATE model-size sweep on PBMC — loss curves & MI by model size

5 architectures spanning ~1.8M → ~96M transformer-body params, all trained on the largest PBMC size (100k cells) across 10 quality levels. Hyperparameters are held fixed at STATE defaults so the only knob varying is parameter count.

Plots overlay one curve per model size — no box plots.

In [ ]:
# Walk the on-disk layout produced by the model-size sweep:
#   model_sizing/model_sizing_NN/<size>/<quality>/
#     ├─ config.json                                               (arch + metadata; written pre-train)
#     ├─ result.json                                               (post-run; status, train_time_s)
#     ├─ checkpoints/**/version_*/metrics.csv                      (Lightning train/val loss curves)
#     └─ MI/<seed>/Y_<signal>_<quality>/lmi_mutual_information.txt
#
# Per-step training and validation loss curves come from Lightning's CSVLogger.
# The DataFrame holds scalars + arch fields; raw curves live in CURVES.
import json
import re
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path('/home/igor/noise_scaling/data/other/model_sizing')
MI_SEED = 42  # first seed emitted by latentmi
ARCH_KEYS = ('emsize', 'd_hid', 'nhead', 'nlayers', 'output_dim', 'pad_length')

def _num(s: str):
    try: return int(s)
    except ValueError:
        try: return float(s)
        except ValueError: return s

def _read_metrics(leaf: Path):
    """Return (train_df, val_df) from Lightning's metrics.csv, or (None, None)."""
    matches = sorted(leaf.glob('checkpoints/**/version_*/metrics.csv'))
    if not matches:
        return None, None
    m = pd.read_csv(matches[0])
    tcol, vcol = 'trainer/train_loss', 'validation/val_loss'
    train_df = (m.dropna(subset=[tcol])[['step', tcol]].rename(columns={tcol: 'train_loss'})
                 .sort_values('step').reset_index(drop=True)) if tcol in m.columns else None
    val_df   = (m.dropna(subset=[vcol])[['step', vcol]].rename(columns={vcol: 'val_loss'})
                 .sort_values('step').reset_index(drop=True)) if vcol in m.columns else None
    return train_df, val_df

def _approx_params(arch: dict) -> float:
    """Approximate transformer-body params (excludes shared pe_embedding).
    Per layer ≈ 4*emsize^2 (attention) + 2*emsize*d_hid (FFN)."""
    e, h, L = arch.get('emsize', 0), arch.get('d_hid', 0), arch.get('nlayers', 0)
    return L * (4 * e * e + 2 * e * h)

rows = []
CURVES: dict = {}  # (trial_id, size, quality) -> {'train': DataFrame, 'val': DataFrame}

for trial_dir in sorted(OUTPUT_DIR.glob('model_sizing_*')):
    if not trial_dir.is_dir():
        continue
    m = re.match(r'model_sizing_(\d+)$', trial_dir.name)
    if not m:
        continue
    trial_id = int(m.group(1))

    for size_dir in sorted(p for p in trial_dir.iterdir() if p.is_dir()):
        size_val = _num(size_dir.name)
        if not isinstance(size_val, (int, float)):
            continue
        for q_dir in sorted(p for p in size_dir.iterdir() if p.is_dir()):
            quality_val = _num(q_dir.name)
            if not isinstance(quality_val, (int, float)):
                continue

            cfg_path = q_dir / 'config.json'
            cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
            arch = cfg.get('arch') or {}
            row = {
                'trial_id': trial_id,
                'trial_name': cfg.get('trial_name', f'model_sizing_{trial_id:02d}'),
                'size': int(size_val),
                'quality': float(quality_val),
                **{k: arch.get(k) for k in ARCH_KEYS},
                'approx_params': _approx_params(arch),
            }

            res_path = q_dir / 'result.json'
            if res_path.exists():
                try:
                    res = json.loads(res_path.read_text())
                    row['status'] = res.get('status', 'unknown')
                    row['train_time_s'] = res.get('train_time_s')
                except json.JSONDecodeError:
                    row['status'] = 'corrupt'
            else:
                row['status'] = 'pending'

            train_df, val_df = _read_metrics(q_dir)
            if train_df is not None and len(train_df):
                row['train_loss_final'] = float(train_df['train_loss'].iloc[-1])
                row['n_train_steps'] = int(train_df['step'].iloc[-1])
            if val_df is not None and len(val_df):
                row['val_loss_final'] = float(val_df['val_loss'].iloc[-1])
            CURVES[(trial_id, int(size_val), float(quality_val))] = {
                'train': train_df, 'val': val_df,
            }

            mi_root = q_dir / 'MI' / str(MI_SEED)
            if mi_root.is_dir():
                for sig_dir in filter(Path.is_dir, mi_root.iterdir()):
                    sm = re.match(r'Y_(.+)_[0-9][0-9_.]*$', sig_dir.name)
                    signal = sm.group(1) if sm else sig_dir.name
                    f = sig_dir / 'lmi_mutual_information.txt'
                    if f.exists():
                        try: row[f'mi_{signal}'] = float(f.read_text().strip())
                        except ValueError: pass
            rows.append(row)

df = pd.DataFrame(rows)
if len(df):
    df = df.sort_values(['trial_id', 'size', 'quality']).reset_index(drop=True)
mi_cols_found = sorted(c for c in df.columns if c.startswith('mi_'))
n_trials = df['trial_id'].nunique() if len(df) else 0
n_ok = int((df['status'] == 'ok').sum()) if 'status' in df.columns else 0
n_curves = sum(1 for v in CURVES.values() if v['train'] is not None and len(v['train']))
print(f'{len(df)} rows across {n_trials} model-size configs  ok={n_ok}  curves={n_curves}  mi_cols={mi_cols_found}')

# Per-trial label: 'M0  e192/L4  (1.8M)'  — used in legends below.
TRIAL_LABEL = {}
for tid, g in df.groupby('trial_id'):
    r0 = g.iloc[0]
    TRIAL_LABEL[int(tid)] = (
        f"M{int(tid)}  e{int(r0['emsize'])}/L{int(r0['nlayers'])}  "
        f"({r0['approx_params']/1e6:.1f}M)"
    )
df

In [ ]:
# MI vs quality, one line per model-size config (overlay).
# Plots all signals side by side so we can see whether bigger models help at every quality.
import matplotlib.pyplot as plt

mi_cols = [c for c in df.columns if c.startswith('mi_')]
if not mi_cols:
    print('No MI columns found — nothing to plot.')
else:
    trial_ids = sorted(df['trial_id'].unique())
    cmap = plt.get_cmap('viridis')
    trial_color = {tid: cmap(i / max(1, len(trial_ids) - 1)) for i, tid in enumerate(trial_ids)}

    fig, axes = plt.subplots(1, len(mi_cols), figsize=(6.5 * len(mi_cols), 4.5),
                              squeeze=False)
    for j, mi_col in enumerate(mi_cols):
        ax = axes[0, j]
        for tid in trial_ids:
            sub = df[(df['trial_id'] == tid) & df[mi_col].notna()].sort_values('quality')
            if sub.empty:
                continue
            ax.plot(sub['quality'].values, sub[mi_col].values,
                    marker='o', linewidth=1.8, color=trial_color[tid],
                    label=TRIAL_LABEL[tid])
        ax.set_xscale('log')
        ax.set_xlabel('quality (downsampling factor)')
        ax.set_ylabel(mi_col.removeprefix('mi_'))
        ax.set_title(f'MI: {mi_col.removeprefix("mi_")} vs quality')
        ax.grid(alpha=0.3)
    axes[0, -1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                       fontsize=9, title='model size')
    plt.tight_layout()
    plt.show()

In [ ]:
# Train / validation loss curves: 2 columns (train, val), 1 row per quality.
# Each subplot overlays one line per model-size config, colored by parameter count.
import matplotlib.pyplot as plt
import numpy as np

qualities = sorted(df['quality'].unique())
trial_ids = sorted(df['trial_id'].unique())
cmap = plt.get_cmap('viridis')
trial_color = {tid: cmap(i / max(1, len(trial_ids) - 1)) for i, tid in enumerate(trial_ids)}

n_rows = len(qualities)
fig, axes = plt.subplots(n_rows, 2, figsize=(12, 2.6 * n_rows),
                          sharex='col', squeeze=False)

for r, q in enumerate(qualities):
    ax_tr, ax_va = axes[r, 0], axes[r, 1]
    for tid in trial_ids:
        keys = [k for k in CURVES if k[0] == tid and np.isclose(k[2], q)]
        for key in keys:
            curves = CURVES[key]
            tr, va = curves['train'], curves['val']
            color = trial_color[tid]
            label = TRIAL_LABEL[tid] if r == 0 else None
            if tr is not None and len(tr):
                ax_tr.plot(tr['step'].values, tr['train_loss'].values,
                           color=color, linewidth=1.2, alpha=0.9, label=label)
            if va is not None and len(va):
                ax_va.plot(va['step'].values, va['val_loss'].values,
                           color=color, linewidth=1.2, alpha=0.9,
                           marker='o', markersize=3, label=label)
    ax_tr.set_ylabel(f'q={q:.4g}\nloss')
    ax_tr.grid(alpha=0.3)
    ax_va.grid(alpha=0.3)

axes[0, 0].set_title('train_loss')
axes[0, 1].set_title('val_loss')
axes[-1, 0].set_xlabel('optimizer step')
axes[-1, 1].set_xlabel('optimizer step')

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
               fontsize=9, title='model size')
fig.suptitle('STATE PBMC model-size sweep: loss curves per quality (one line per model size)', y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Final loss vs model size: one line per quality, x = approx params (log).
# Quick read on whether scaling helps and where it saturates per quality bucket.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), squeeze=False)
qualities = sorted(df['quality'].unique())
qmap = plt.get_cmap('plasma')
qcolor = {q: qmap(i / max(1, len(qualities) - 1)) for i, q in enumerate(qualities)}

for j, (loss_col, title) in enumerate([('train_loss_final', 'final train_loss'),
                                        ('val_loss_final',   'final val_loss')]):
    ax = axes[0, j]
    if loss_col not in df.columns:
        ax.set_title(f'{title} (missing)')
        continue
    for q in qualities:
        sub = (df[(df['quality'] == q) & df[loss_col].notna()]
                 .sort_values('approx_params'))
        if sub.empty:
            continue
        ax.plot(sub['approx_params'].values, sub[loss_col].values,
                marker='o', linewidth=1.5, color=qcolor[q], label=f'q={q:.4g}')
    ax.set_xscale('log')
    ax.set_xlabel('transformer-body params (approx)')
    ax.set_ylabel(loss_col)
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0, -1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, title='quality')
plt.tight_layout()
plt.show()